# KuchoLM training — 7M copy curriculum

7M のまま、コピー崩壊・反復・EOS未学習を抑える版です。

- 12k BPE + byte fallback
- 50k examples
- COPY warmup 1 pass
- NIDA + COPY mixed 2 epochs
- EOS loss 4x
- label smoothingなし
- 各epochで固定5文を生成・checkpoint保存
- no-repeat 3-gram


In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece torch


## 1. 設定


In [ ]:
from pathlib import Path
import difflib, json, math, random, re, string

import MeCab
import sentencepiece as spm
import torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import Dataset, DataLoader

DATA_PATH = Path('/content/kucholm_nida.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS = 50_000
VOCAB_SIZE = 12_000
MAX_LEN = 160
COPY_RATIO = 0.50
REF_RATIO = 0.25
COPY_WARMUP_ROWS = 30_000
MIXED_EPOCHS = 2
EOS_WEIGHT = 4.0
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tagger = MeCab.Tagger()
print('device:', device)


## 2. NIDA_FICTION データ生成


In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
SENTENCE_RE = re.compile(r'(.+?[。！？!?]+|.+$)', re.S)

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            f = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': f[0] if len(f) > 0 else '',
                'ctype': f[4] if len(f) > 4 else '*',
                'lemma': f[7] if len(f) > 7 else '*',
                'orth_base': f[10] if len(f) > 10 else '*',
            })
        node = node.next
    return tokens

def dictionary_form(token):
    for key in ('orth_base', 'lemma'):
        value = token.get(key, '*')
        if value not in ('', '*') and re.search(r'[ぁ-ん一-龯]', value):
            return value
    return token['surface']

def is_ichidan(token, base):
    ctype = token.get('ctype', '')
    return '下一段' in ctype or '上一段' in ctype or '一段' in ctype

def ta_form(base, token):
    if base == '行く': return '行った'
    if base == '来る': return '来た'
    if base == 'する': return 'した'
    if is_ichidan(token, base): return base[:-1] + 'た'
    if base.endswith(('う','つ','る')): return base[:-1] + 'った'
    if base.endswith(('む','ぶ','ぬ')): return base[:-1] + 'んだ'
    if base.endswith('く'): return base[:-1] + 'いた'
    if base.endswith('ぐ'): return base[:-1] + 'いだ'
    if base.endswith('す'): return base[:-1] + 'した'
    return base + 'た'

def nai_form(base, token):
    if base == 'する': return 'しない'
    if base == '来る': return '来ない'
    if is_ichidan(token, base): return base[:-1] + 'ない'
    if base.endswith('う'): return base[:-1] + 'わない'
    table = {'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    return base[:-1] + table[base[-1]] + 'ない' if base[-1:] in table else base + 'ない'

def convert_polite_tail(body):
    for pattern, replacement in [
        (r'ということでした$', 'ってことだった'),
        (r'ということです$', 'ってこと'),
        (r'かもしれません$', 'かもしれない'),
        (r'わかりません$', 'わからない'),
        (r'知りません$', '知らない'),
        (r'いけません$', 'いけない'),
        (r'ありません$', 'ない'),
        (r'ございました$', 'あった'),
        (r'ございます$', 'ある'),
        (r'てきました$', 'てきた'),
        (r'て来ました$', 'て来た'),
        (r'てしまいました$', 'てしまった'),
        (r'でしまいました$', 'でしまった'),
        (r'でした$', 'だった'),
        (r'です$', ''),
    ]:
        if re.search(pattern, body):
            return re.sub(pattern, replacement, body)

    tokens = parse_tokens(body)
    surfaces = [t['surface'] for t in tokens]
    for suffix, mode in [
        (['ませ','ん','でし','た'], 'negative_past'),
        (['ませ','ん'], 'negative'),
        (['まし','た'], 'past'),
        (['ます'], 'present'),
    ]:
        if len(surfaces) < len(suffix) or surfaces[-len(suffix):] != suffix:
            continue
        end = len(tokens) - len(suffix)
        vi = next((i for i in range(end - 1, -1, -1) if tokens[i]['pos'] == '動詞'), None)
        if vi is None:
            continue
        verb = tokens[vi]
        base = dictionary_form(verb)
        prefix = ''.join(t['surface'] for t in tokens[:vi])
        if mode == 'present':
            replacement = base
        elif mode == 'past':
            replacement = ta_form(base, verb)
        else:
            negative = nai_form(base, verb)
            replacement = negative if mode == 'negative' else negative[:-2] + 'なかった'
        return prefix + replacement
    return body

def convert_sentence(sentence):
    m = re.match(r'^(\s*)(.*?)(\s*)$', sentence, re.S)
    leading, core, trailing = m.groups()
    if not core or URL_RE.search(core):
        return sentence

    pm = re.search(r'([。！？!?]+)$', core)
    punctuation = pm.group(1) if pm else ''
    body = core[:-len(punctuation)] if punctuation else core
    is_question = bool(re.search(r'[？?]$', punctuation))

    converted = convert_polite_tail(body)
    if converted.endswith(('ね','よ','な')):
        converted = converted[:-1] + 'ニダ' + converted[-1]
    else:
        converted += 'ニカ' if is_question else 'ニダよ'
    return leading + converted + punctuation + trailing

def to_nida(text):
    if not text or URL_RE.search(text):
        return None
    return ''.join(convert_sentence(m.group(0)) for m in SENTENCE_RE.finditer(text))

if not DATA_PATH.exists():
    dataset = load_dataset('range3/cc100-ja', split='train', streaming=True)
    written = 0
    with DATA_PATH.open('w', encoding='utf-8') as out:
        for row in dataset:
            source = str(row['text'])
            if len(source.strip()) < 2 or len(source) > 220:
                continue
            target = to_nida(source)
            if not target or target == source:
                continue
            out.write(json.dumps({'source': source, 'target': target}, ensure_ascii=False) + '\n')
            written += 1
            if written >= MAX_ROWS:
                break
    print('written:', written)
else:
    print('using existing:', DATA_PATH)


## 3. COPY curriculum + SentencePiece


In [ ]:
raw_rows = []
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        x = json.loads(line)
        raw_rows.append((x['source'], x['target']))

random.shuffle(raw_rows)
raw_rows = raw_rows[:MAX_ROWS]
cut = max(1, int(len(raw_rows) * 0.98))
train_raw, val_raw = raw_rows[:cut], raw_rows[cut:]

RARE_CHARS = '髙﨑𠮷神邉邊齋齊塚'
ASCII_POOL = string.ascii_uppercase + string.digits

def make_ref():
    a = ''.join(random.choices(ASCII_POOL, k=7))
    b = ''.join(random.choices(ASCII_POOL, k=5))
    return f'[REF:{a}-{b}/{random.choice(RARE_CHARS)}{random.choice(RARE_CHARS)}]'

nida_rows = []
copy_rows = []
for source, target in train_raw:
    nida_rows.append((f'<NIDA_FICTION> {source}', target))
    copy_rows.append((f'<COPY> {source}', source))
    if random.random() < REF_RATIO:
        ref = make_ref()
        nida_rows.append((f'<NIDA_FICTION> {ref} {source}', f'{ref} {target}'))

mixed_rows = nida_rows + random.sample(copy_rows, min(len(copy_rows), int(len(nida_rows) * COPY_RATIO)))
random.shuffle(mixed_rows)

spm_input = WORK_DIR / 'spm_train.txt'
with spm_input.open('w', encoding='utf-8') as f:
    for source, target in mixed_rows:
        f.write(source.replace('\n', ' ') + '\n')
        f.write(target.replace('\n', ' ') + '\n')

spm.SentencePieceTrainer.train(
    input=str(spm_input),
    model_prefix=str(WORK_DIR / 'kucholm_spm'),
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    character_coverage=1.0,
    byte_fallback=True,
    normalization_rule_name='identity',
    split_digits=True,
    hard_vocab_limit=False,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    user_defined_symbols=['<NIDA_FICTION>', '<COPY>'],
)

sp = spm.SentencePieceProcessor(model_file=str(WORK_DIR / 'kucholm_spm.model'))
PAD, UNK, BOS, EOS = 0, 1, 2, 3
VOCAB = sp.vocab_size()
print('vocab:', VOCAB)


## 4. Dataset / Model


In [ ]:
def encode(text):
    return [BOS] + sp.encode(text, out_type=int) + [EOS]

def fits(pair):
    return len(encode(pair[0])) <= MAX_LEN and len(encode(pair[1])) <= MAX_LEN

copy_rows = [p for p in copy_rows if fits(p)]
mixed_rows = [p for p in mixed_rows if fits(p)]
val_rows = [(f'<NIDA_FICTION> {s}', t) for s, t in val_raw]
val_rows = [p for p in val_rows if fits(p)]

class PairDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        source, target = self.rows[i]
        return torch.tensor(encode(source)), torch.tensor(encode(target))

def collate(batch):
    source, target = zip(*batch)
    return (
        nn.utils.rnn.pad_sequence(source, batch_first=True, padding_value=PAD),
        nn.utils.rnn.pad_sequence(target, batch_first=True, padding_value=PAD),
    )

BATCH = 64 if device.type == 'cuda' else 8
loader_args = {
    'batch_size': BATCH,
    'collate_fn': collate,
    'pin_memory': device.type == 'cuda',
    'num_workers': 2 if device.type == 'cuda' else 0,
}
copy_loader = DataLoader(PairDataset(copy_rows[:COPY_WARMUP_ROWS]), shuffle=True, **loader_args)
mixed_loader = DataLoader(PairDataset(mixed_rows), shuffle=True, **loader_args)
val_loader = DataLoader(PairDataset(val_rows), shuffle=False, **loader_args)

D_MODEL = 224
NHEAD = 8
ENC_LAYERS = 3
DEC_LAYERS = 3
FF = 896

class KuchoTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.pos = nn.Embedding(MAX_LEN, D_MODEL)
        self.tf = nn.Transformer(
            d_model=D_MODEL,
            nhead=NHEAD,
            num_encoder_layers=ENC_LAYERS,
            num_decoder_layers=DEC_LAYERS,
            dim_feedforward=FF,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.lm_head = nn.Linear(D_MODEL, VOCAB, bias=False)
        self.lm_head.weight = self.embed.weight

    def add_pos(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device).unsqueeze(0)
        return self.embed(ids) * math.sqrt(D_MODEL) + self.pos(pos)

    def forward(self, source, target):
        mask = nn.Transformer.generate_square_subsequent_mask(target.size(1), device=target.device)
        hidden = self.tf(
            self.add_pos(source),
            self.add_pos(target),
            tgt_mask=mask,
            src_key_padding_mask=source.eq(PAD),
            tgt_key_padding_mask=target.eq(PAD),
            memory_key_padding_mask=source.eq(PAD),
        )
        return self.lm_head(hidden)

model = KuchoTransformer().to(device)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.3f}M parameters')
print('rows:', len(copy_rows[:COPY_WARMUP_ROWS]), len(mixed_rows), len(val_rows), 'batch:', BATCH)


## 5. 推論 / 学習


In [ ]:
TESTS = [
    '今日は学校です。',
    '明日は雨が降るかもしれません。',
    '最近少し暖かくなってきました。',
    '製品KuchoLM-X7-2026は正常に動作しています。',
    '髙﨑𠮷野家ABC-123を確認しました。',
]

@torch.no_grad()
def infer(text, tag='<NIDA_FICTION>'):
    source_ids = encode(f'{tag} {text}')
    source = torch.tensor([source_ids], device=device)
    output = [BOS]
    limit = min(MAX_LEN - 1, len(source_ids) + 12)

    for step in range(limit):
        logits = model(source, torch.tensor([output], device=device))[0, -1].clone()

        if len(output) >= 3:
            prefix = tuple(output[-2:])
            banned = {
                output[i + 2]
                for i in range(len(output) - 2)
                if tuple(output[i:i + 2]) == prefix
            }
            if banned:
                logits[list(banned)] = -float('inf')

        if len(output) >= 2 and output[-1] == output[-2]:
            logits[output[-1]] = -float('inf')

        if step >= max(4, len(source_ids) - 4):
            logits[EOS] += 0.6

        next_id = int(torch.argmax(logits))
        if next_id == EOS:
            break
        output.append(next_id)

    return sp.decode(output[1:])

def show_samples(label):
    model.eval()
    print('\n---', label, '---')
    for sample in TESTS:
        print(sample, '->', infer(sample))

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.98), weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

def loss_fn(logits, target):
    y = target.reshape(-1)
    losses = nn.functional.cross_entropy(
        logits.reshape(-1, VOCAB),
        y,
        ignore_index=PAD,
        reduction='none',
    )
    valid = y != PAD
    weights = torch.ones_like(losses)
    weights[y == EOS] = EOS_WEIGHT
    return (losses[valid] * weights[valid]).sum() / weights[valid].sum()

def train_one(loader):
    model.train()
    total = 0.0
    for source, target in loader:
        source = source.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(source, target[:, :-1])
            loss = loss_fn(logits, target[:, 1:])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / max(1, len(loader))

@torch.no_grad()
def validation_loss():
    model.eval()
    total = 0.0
    for source, target in val_loader:
        source = source.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        total += loss_fn(model(source, target[:, :-1]), target[:, 1:]).item()
    return total / max(1, len(val_loader))

print('COPY warmup:', train_one(copy_loader))
torch.save({'model': model.state_dict()}, WORK_DIR / 'copy_warmup.pt')
show_samples('COPY warmup')

best_val = float('inf')
best_path = WORK_DIR / 'KuchoLM-NIDA-7M.pt'

for epoch in range(1, MIXED_EPOCHS + 1):
    train_loss = train_one(mixed_loader)
    val_loss = validation_loss()

    epoch_path = WORK_DIR / f'KuchoLM-NIDA-7M-epoch{epoch}.pt'
    torch.save({'model': model.state_dict(), 'val': val_loss}, epoch_path)
    print(f'epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f}')
    show_samples(f'epoch {epoch}')

    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model': model.state_dict(), 'best_val': best_val}, best_path)

model.load_state_dict(torch.load(best_path, map_location=device)['model'])
show_samples('BEST')


## 6. 評価


In [ ]:
def similarity(expected, actual):
    return difflib.SequenceMatcher(None, expected, actual).ratio()

scores = []
for tagged_source, expected in val_rows[:100]:
    source = tagged_source.removeprefix('<NIDA_FICTION> ')
    scores.append(similarity(expected, infer(source)))

copy_tests = []
for source, target in val_raw[:100]:
    ref = make_ref()
    tagged = f'{ref} {source}'
    if len(encode(f'<NIDA_FICTION> {tagged}')) <= MAX_LEN:
        copy_tests.append((tagged, ref))

marker_ok = sum(ref in infer(source) for source, ref in copy_tests)

print('char similarity:', sum(scores) / max(1, len(scores)))
print('REF copy accuracy:', marker_ok / max(1, len(copy_tests)))
print('model:', best_path)
print('tokenizer:', WORK_DIR / 'kucholm_spm.model')
